In [ ]:
!pip install -q fair-esm pyarrow

In [ ]:
import torch, esm, numpy as np, pandas as pd, hashlib, json, time
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
model = model.eval().cuda().half()
conv = alphabet.get_batch_converter()

def embed(seqs, batch=512):
    seqs = list(seqs)
    order = np.argsort([len(s) for s in seqs], kind="stable")
    out = np.zeros((len(seqs), 1280), dtype=np.float32)
    t0 = time.time()
    for i in range(0, len(seqs), batch):
        idx = order[i:i + batch]
        _, _, toks = conv([(str(k), seqs[k]) for k in idx])
        with torch.no_grad():
            reps = model(toks.cuda(), repr_layers=[33])["representations"][33].float()
        lens = torch.tensor([len(seqs[k]) for k in idx], device=reps.device)
        pos = torch.arange(reps.shape[1], device=reps.device)[None, :]
        mask = ((pos >= 1) & (pos <= lens[:, None])).unsqueeze(-1)   # residues only, no BOS/EOS/PAD
        out[idx] = ((reps * mask).sum(1) / lens[:, None]).cpu().numpy()
        if (i // batch) % 20 == 0:
            print(f"{i}/{len(seqs)}  {time.time()-t0:.0f}s")
    assert np.isfinite(out).all(), "NaN/Inf in embeddings"
    return out

In [ ]:
cand = pd.read_parquet("candidates.parquet")
assert cand["sequence"].is_unique
seqs = cand["sequence"].tolist()

t = time.time(); _ = embed(seqs[:2000]); dt = time.time() - t
print(f"2000 seqs in {dt:.0f}s -> projected full run {dt * len(seqs) / 2000 / 60:.1f} min")

t = time.time(); E = embed(seqs); runtime = time.time() - t
np.save("c4_esm2_t33_650M.npy", E.astype(np.float16))
pd.DataFrame({"row": range(len(seqs)), "peptide_id": cand["peptide_id"], "sequence": seqs}) \
  .to_parquet("c4_embedding_index.parquet")
json.dump({"model": "esm2_t33_650M_UR50D", "layer": 33, "pooling": "mean over residues, excl BOS/EOS/PAD",
           "dtype_saved": "float16", "dim": 1280, "n": len(seqs), "device": torch.cuda.get_device_name(0),
           "runtime_s": round(runtime), "row_id_sha256": hashlib.sha256("\n".join(seqs).encode()).hexdigest()},
          open("c4_embedding_manifest.json", "w"), indent=2)
print(E.shape)

In [ ]:
rng = np.random.default_rng(42)
aa = list("ACDEFGHIKLMNPQRSTVWY")
w = np.ones(20)
for a in "WYHMCP": w[aa.index(a)] = 3.0     # redox/aromatic/proline-rich bias
for a in "LIVFA":  w[aa.index(a)] = 2.0     # hydrophobic bias
w /= w.sum()

pool = set(cand["sequence"])                 # cloud must not come from the candidate pool
rows = set()
while len(rows) < 500:
    s = "".join(rng.choice(aa, size=int(rng.integers(3, 21)), p=w))
    if len(set(s)) >= 2 and s not in pool:
        rows.add(s)

ref = pd.DataFrame({"id": [f"MOCK{i:04d}" for i in range(500)], "sequence": sorted(rows),
                    "evidence_tier": "MOCK", "source": "mock_cloud", "is_plant": False})
ref.to_csv("ref_cloud_sequences.csv", index=False)
R = embed(ref["sequence"].tolist())
np.save("ref_cloud.npy", R.astype(np.float16))
print(R.shape)

In [ ]:
!zip -q emb.zip c4_esm2_t33_650M.npy c4_embedding_index.parquet c4_embedding_manifest.json ref_cloud.npy ref_cloud_sequences.csv
from google.colab import files; files.download("emb.zip")